In [2]:
# Capa Bronze - BioTIME (crea carpeta bronze_output para Silver)
import duckdb
import pyreadr
import pandas as pd
import os

# ========== NUEVAS RUTAS (absolutas, según tu estructura) ==========
input_dir = 'P:/proyecto_1/datos/datos_crudos'        # archivos .rds y .csv
output_dir = 'P:/proyecto_1/datos/bronze_output'      # salida DuckDB y opcional Parquet
os.makedirs(output_dir, exist_ok=True)

local_db = os.path.join(output_dir, 'biotime_bronze.duckdb')

# Conectar a DuckDB
conn = duckdb.connect(local_db)
conn.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

# DDL para tabla de datos
ddl_data = """
DROP TABLE IF EXISTS bronze.biotime_data;
CREATE TABLE bronze.biotime_data (
    ID_ALL_RAW_DATA INT,
    ABUNDANCE DOUBLE,
    BIOMASS DOUBLE,
    ID_SPECIES VARCHAR(50),
    SAMPLE_DESC VARCHAR(200),
    LATITUDE DOUBLE,
    LONGITUDE DOUBLE,
    DEPTH DOUBLE,
    DAY VARCHAR(10),
    MONTH VARCHAR(10),
    YEAR INT,
    STUDY_ID INT,
    newID INT,
    valid_name VARCHAR(200),
    resolution VARCHAR(50),
    taxon VARCHAR(200),
    ingestion_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

ddl_meta = """
DROP TABLE IF EXISTS bronze.biotime_metadata;
CREATE TABLE bronze.biotime_metadata (
    STUDY_ID INT,
    REALM VARCHAR(100),
    CLIMATE VARCHAR(100),
    GENERAL_TREAT VARCHAR(500),
    TREATMENT VARCHAR(200),
    TREAT_COMMENTS VARCHAR(1000),
    TREAT_DATE VARCHAR(50),
    CEN_LATITUDE DOUBLE,
    CEN_LONGITUDE DOUBLE,
    HABITAT VARCHAR(200),
    PROTECTED_AREA VARCHAR(100),
    AREA DOUBLE,
    BIOME_MAP VARCHAR(200),
    TAXA VARCHAR(200),
    ORGANISMS VARCHAR(500),
    TITLE VARCHAR(500),
    AB_BIO VARCHAR(200),
    DATA_POINTS INT,
    START_YEAR INT,
    END_YEAR INT,
    CENT_LAT DOUBLE,
    CENT_LONG DOUBLE,
    NUMBER_OF_SPECIES INT,
    NUMBER_OF_SAMPLES INT,
    NUMBER_LAT_LONG INT,
    TOTAL INT,
    GRAIN_SIZE_TEXT VARCHAR(100),
    AREA_SQ_KM DOUBLE,
    CONTACT_1 VARCHAR(200),
    CONTACT_2 VARCHAR(200),
    CONT_1_MAIL VARCHAR(200),
    CONT_2_MAIL VARCHAR(200),
    PERMISSIONS VARCHAR(500),
    WEB_LINK VARCHAR(500),
    DATA_SOURCE VARCHAR(500),
    DATE_TO_DB VARCHAR(50),
    METHODS VARCHAR(1000),
    SUMMARY_METHODS VARCHAR(500),
    LINK_ID DOUBLE,
    COMMENTS VARCHAR(2000),
    DATES_CHANGED VARCHAR(200),
    CURATOR VARCHAR(200),
    LOC_ADDED VARCHAR(200),
    DATE_STUDY_ADDED VARCHAR(50),
    ABUNDANCE_TYPE VARCHAR(100),
    BIOMASS_TYPE VARCHAR(100),
    SAMPLE_DESC_NAME VARCHAR(500),
    ingestion_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

conn.execute(ddl_data)
conn.execute(ddl_meta)

# Cargar datos
rds_path = os.path.join(input_dir, 'biotime_v2_query_15April25.rds')
csv_path = os.path.join(input_dir, 'biotime_v2_metadata_15April25.csv')


result = pyreadr.read_r(rds_path)
df_data = result[None]
df_meta = pd.read_csv(csv_path)
print(f"Datos: {df_data.shape}, Metadatos: {df_meta.shape}")

conn.register('temp_data', df_data)
conn.register('temp_meta', df_meta)


conn.execute("""
INSERT INTO bronze.biotime_data (
    ID_ALL_RAW_DATA, ABUNDANCE, BIOMASS, ID_SPECIES, SAMPLE_DESC,
    LATITUDE, LONGITUDE, DEPTH, DAY, MONTH, YEAR, STUDY_ID, newID,
    valid_name, resolution, taxon
)
SELECT
    ID_ALL_RAW_DATA, ABUNDANCE, BIOMASS, ID_SPECIES, SAMPLE_DESC,
    LATITUDE, LONGITUDE, DEPTH, DAY, MONTH, YEAR, STUDY_ID, newID,
    valid_name, resolution, taxon
FROM temp_data;
""")

conn.execute("""
INSERT INTO bronze.biotime_metadata SELECT *, CURRENT_TIMESTAMP FROM temp_meta;
""")
print("Datos insertados.")

# Validaciones
print("\n--- Validaciones ---")
counts = conn.execute("""
    SELECT 'biotime_data' AS tabla, COUNT(*) AS filas FROM bronze.biotime_data
    UNION ALL
    SELECT 'biotime_metadata', COUNT(*) FROM bronze.biotime_metadata
""").fetchdf()
print(counts)

nulls = conn.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN STUDY_ID IS NULL THEN 1 ELSE 0 END) AS null_study_id,
        SUM(CASE WHEN YEAR IS NULL THEN 1 ELSE 0 END) AS null_year,
        SUM(CASE WHEN valid_name IS NULL THEN 1 ELSE 0 END) AS null_species
    FROM bronze.biotime_data;
""").fetchdf()
print("\nNulos en datos:")
print(nulls)

years = conn.execute("SELECT MIN(YEAR), MAX(YEAR) FROM bronze.biotime_data").fetchdf()
print(f"\nRango de años: {years.iloc[0,0]} - {years.iloc[0,1]}")

# Cerrar conexión
conn.close()
print(f"\n¡Bronce completado! Base DuckDB guardada en: {local_db}")

Datos: (11989233, 16), Metadatos: (708, 47)
Datos insertados.

--- Validaciones ---
              tabla     filas
0      biotime_data  11989233
1  biotime_metadata       708

Nulos en datos:
      total  null_study_id  null_year  null_species
0  11989233            0.0        0.0           0.0

Rango de años: 1874 - 2023

¡Bronce completado! Base DuckDB guardada en: P:/proyecto_1/datos/bronze_output\biotime_bronze.duckdb
